In [ ]:
from pathlib import Path
import pandas as pd 
import numpy as np
from tqdm.auto import tqdm
import plotly.express as px
tqdm.pandas()

root = Path('/data/data/malpolon/xgb/')
inputs_path = Path('/marbec-data/RLS-Australia/malpolon/inputs/australia/')
output_path = Path('/marbec-data/RLS-Australia/malpolon/outputs/')

In [4]:
fulldf = pd.read_csv(inputs_path / 'database_common.csv', index_col='survey_id',
                     dtype = {22:str, 24:str, 25:str})
species = list(fulldf.columns[-818:-1])
groundtruth = fulldf[species]

## P-A

#### Species F1

In [ ]:
xgbbest_f1 = pd.read_csv(next((root / 'pa').glob("xgbbest_f1*.csv")), index_col = 0)

cp = '26_hum_env_dhw_bathy_common_pa-2025-11-10_16-31'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

cp = '37_mm4_transductif-2025-12-02_12-47'
td_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

In [ ]:
scores = pd.concat([xgbbest_f1["f1"], mm_f1["f1"], td_f1["f1"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'F1']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [ ]:
import plotly.express as px

fig = px.line(data[data['x'] <= 400], x='x', y='F1', color='model', template = 'simple_white',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Presence-absence classification',
    xaxis_title='Species rank',
    yaxis_title='F1 score',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)

fig.update_traces(line={'width': 4})

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

x, y = scores[names[0]].values, scores[names[2]].values

# Draw a combo histogram and scatterplot with density contours
f, ax = plt.subplots(figsize=(9, 9))
sns.scatterplot(x=x, y=y, s=5, color=".15")
sns.histplot(x=x, y=y, bins=40, pthresh=.1, cmap="mako")
sns.kdeplot(x=x, y=y, levels=5, color="w", linewidths=1)
sns.lineplot(x=[min(x), max(x)], y=[min(x), max(x)],color='red')
ax.set(xlabel=names[0] + ' F1', ylabel=names[2] + ' F1')

#### Site F1

In [ ]:
xgbbest_f1 = pd.read_csv(next((root / 'pa').glob("xgbbest_f1_by_site.csv")), index_col = 0)

cp = '37_mm4_transductif-2025-12-02_12-47'
td_f1 = pd.read_csv(next((output_path / cp).glob("f1_by_site_new.csv")), index_col = 0)

scores = pd.concat([xgbbest_f1['f1'], td_f1['f1']], axis=1)
scores.columns = ['XGB', 'TD']

In [ ]:
import plotly.express as px

site_F1_improv = pd.DataFrame(f1scores)[['site', 'lat', 'lon']]
#site_F1_improv['delta'] = 100*(scores['TD'] - scores['XGB']) / (scores['XGB']+1e-6)
site_F1_improv['delta'] = (scores['TD'] - scores['XGB'])

fig = px.scatter_map(site_F1_improv, lat='lat',lon='lon', color='delta', height = 1100, width = 1200,
                     center={'lat':-28, 'lon':133}, zoom=4, opacity=0.5,
                     color_continuous_midpoint=0,
                     color_continuous_scale=px.colors.sequential.RdBu, map_style='dark')

fig.update_traces(marker=dict(size=20))
fig.show()

In [ ]:
site_F1_improv['delta'].describe()

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

x, y = scores['XGB'].values, scores['TD'].values

# Draw a combo histogram and scatterplot with density contours
f, ax = plt.subplots(figsize=(9, 9))
sns.scatterplot(x=x, y=y, s=5, color=".15")
sns.histplot(x=x, y=y, bins=40, pthresh=.1, cmap="mako")
sns.kdeplot(x=x, y=y, levels=5, color="w", linewidths=1)
sns.lineplot(x=[min(x), max(x)], y=[min(x), max(x)],color='red')
ax.set(xlabel='XGB F1', ylabel='TD F1')

## Reg

#### Species R2

In [ ]:
xgbbest_f1 = pd.read_csv(next((root / 'reg').glob("xgb_best_r2*.csv")), index_col = 0)

cp = '38_mm4_transductif_reg-2026-01-07_16-44'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '38_mm4_transductif_reg-2025-12-17_15-41'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

In [ ]:
scores = pd.concat([xgbbest_f1["pearsonr"], mm_f1["pearsonr"], td_f1["pearsonr"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'pearsonr']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [ ]:
import plotly.express as px

fig = px.line(data[data['x'] <= 400], x='x', y='pearsonr', color='model', template = 'simple_white',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Biomass regression',
    xaxis_title='Species rank',
    yaxis_title='Pearson R',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)

fig.update_traces(line={'width': 4})

In [ ]:
scores['delta'] = 100*(scores['Transductive'] - scores['XGBoost']) / (scores['XGBoost']+1e-6)
scores['delta'].describe()

## Traits based

#### Load traits

In [ ]:
traits = pd.read_csv('/data/data/RLS/traits.csv', index_col=0)

troph_level_dict = {s: traits['Troph'].get(s, np.nan) for s in species}
troph_level = pd.Series(troph_level_dict, index = species, name = 'troph_level')

uicnL_status_dict = {s: traits['IUCN_inferred_Loiseau23'].get(s, 'No Status') for s in species}
uicnL_status = pd.Series(uicnL_status_dict, index = species, name = 'uicnL_status')
uicnL_threatened = pd.Series([(uicnL_status_dict[s] == 'Threatened')*1 for s in species], index = species, name = 'Threatened')

uicn_species = list(uicnL_threatened[uicnL_threatened == 1].index)

#### Load scores to compare

In [ ]:
# P-A

xgb_f1 = pd.read_csv(next((root / 'pa').glob("xgbbest_f1--*.csv")), index_col = 0)

cp = '26_hum_env_dhw_bathy_common_pa-2025-11-10_16-31'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

cp = '37_mm4_transductif-2025-12-02_12-47'
td_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

In [ ]:
# Reg

cp = '38_mm4_transductif_reg-2025-12-17_18-53'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '38_mm4_transductif_reg-2026-01-07_16-44'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

In [ ]:
# Calculate progress

scores = pd.concat([xgb_f1["f1"], td_f1["f1"]], axis = 1)
names = ['XGB', 'Transductive']
scores.columns = names
scores['progress'] = (scores[names[1]] - scores[names[0]])
scores['max'] = scores[names].max(axis=1)
relevant_scores = scores.loc[scores['max'] > 0.4]

In [ ]:
# Add traits to scores

traitname = 'IUCN_inferred_Loiseau23' # 'DemersPelag' 'Troph' 'IUCN_inferred_Loiseau23'

relevant_scores['uicnL_status'] = uicnL_status.loc[relevant_scores.index]
relevant_scores['troph_level'] = troph_level.loc[relevant_scores.index]

#### Plot performance increase vs traits

In [ ]:



fig = px.violin(relevant_scores, x=traitname, y="progress", points = 'all', color = traitname, color_discrete_sequence= px.colors.qualitative.Pastel)


fig.update_layout(
    template='simple_white',
    width = 800, height=800,
    yaxis_title='Pearson delta',
    xaxis_title='',
    font=dict(size=20),
    showlegend=False)
    

fig.add_shape(type="line",
    xref="paper",
    x0=0, y0=0,
    x1=1, y1=0,
    line=dict(
        color="blue",
        width=3,
    ))

fig.show()

## Eco indicators

In [ ]:
xgb_f1 = pd.read_csv(next((root / 'eco').glob("xgb_best_r2*.csv")), index_col = 0)

cp = '40_mm4_transductif_eco-2026-01-23_17-45'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '40_mm4_transductif_eco-2026-01-23_16-56'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

tdstacked_f1 = pd.read_csv('~/eco_indic_corr.csv', index_col=0)

In [ ]:
scores = pd.concat([xgb_f1["pearsonr"], mm_f1["pearsonr"], td_f1["pearsonr"], tdstacked_f1["pearsonr"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive (direct)', 'Transductive (computed)']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'pearsonr']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [ ]:
import plotly.express as px

fig = px.bar(data, x=data['species'], y='pearsonr', color='model', template = 'simple_white', barmode='group',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Ecological indicators regression',
    xaxis_title='',
    yaxis_title='Pearson R',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)


## Any 2

#### 1v1 BlandAltman

In [ ]:
import pyCompare

pyCompare.blandAltman(scores[names[3]].values,scores[names[0]].values)

#### 1v1 Scatter plot

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

x, y = scores[names[0]].values, scores[names[3]].values

# Draw a combo histogram and scatterplot with density contours
f, ax = plt.subplots(figsize=(9, 9))
sns.scatterplot(x=x, y=y, s=5, color=".15")
sns.histplot(x=x, y=y, bins=40, pthresh=.1, cmap="mako")
sns.kdeplot(x=x, y=y, levels=5, color="w", linewidths=1)
sns.lineplot(x=[min(x), max(x)], y=[min(x), max(x)],color='red')
ax.set(xlabel=names[0] + ' F1', ylabel=names[3] + ' F1')

#### 1v1 arrow scatter

In [ ]:
scores['av'] = np.maximum(scores[names[0]], scores[names[3]])

best_scores = scores[scores['av'] > 0.3]
(best_scores[names[3]] - best_scores[names[2]]).hist(bins=40)

best_scores = scores.sort_values(by='XGBoost', ascending=False)
best_scores = best_scores[best_scores['av'] > 0.4]
best_scores['x'] = np.arange(len(best_scores))

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Example data
x = best_scores['x']
y1 = best_scores['XGBoost']
y2 = best_scores['Transductif']
hover_text = best_scores.index.to_list()

# Create the figure
fig = go.Figure()

# Add the two series as scatter plots
fig.add_trace(go.Scatter(
    x=x, y=y1,
    mode='markers',
    marker=dict(color=px.colors.qualitative.Pastel[0], size=10),
    name='XGBoost',
    hovertext=hover_text,
    hoverinfo='text'
))

fig.add_trace(go.Scatter(
    x=x, y=y2,
    mode='markers',
    marker=dict(color=px.colors.qualitative.Pastel[3], size=10),
    name='Transductive',
    hovertext=hover_text,
    hoverinfo='text'
))

# Add arrows for the differences
for i in range(len(x)):
    fig.add_annotation(
        ax=x[i], ay=y1[i],
        axref='x', ayref='y',
        x=x[i], y=y2[i],
        xref='x', yref='y',
        showarrow=True,
        arrowhead=1,
        arrowsize=1.5,
        arrowcolor=px.colors.qualitative.Pastel1[1],
        arrowwidth=1.5,
    )


fig.update_layout(
    template='simple_white',
    width = 1200, height=800,
    xaxis_title='Species rank',
    yaxis_title='F1 score',
    font=dict(size=20),
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)
